# Representative days

Cluster six synthetic days of demand and temperature into two observed profiles. Each day has four six-hour samples. Install Vael in the notebook kernel environment before running these cells.


In [ ]:
"""Cluster six synthetic days and print framework-independent JSON.

Each day contains four six-hour samples of demand and temperature. Keeping the
features separate preserves their order and units in the output. Run this file
from an installed Vael environment.
"""


import json

import numpy as np

from vael import AHCResult, ahc



def representative_days() -> AHCResult:
    """Reduce two groups of three identical days to two observed profiles."""
    demand = np.concatenate(
        [np.tile([10, 12, 14, 10], 3), np.tile([20, 24, 28, 20], 3)],
    )
    temperature = np.concatenate(
        [np.tile([15, 18, 22, 16], 3), np.tile([25, 28, 32, 26], 3)],
    )
    return ahc(
        demand,
        temperature,
        periods=2,
        period_length=4,
        selection="nearest_centroid",
    )




result = representative_days()
print("Representative indices:", result.representative_indices)
print("Weights:", result.weights)
print("Labels:", result.labels)


## Export and reconstruct

The two groups contain three identical days each, so the weights are `[3, 3]`. A consumer can read the JSON without depending on the result class. The feature order remains demand, then temperature.


In [ ]:

payload = json.loads(json.dumps(result.to_dict()))
profiles = np.asarray(payload["representatives"])
reconstructed = profiles[payload["labels"]].reshape(-1, profiles.shape[-1])
np.testing.assert_array_equal(reconstructed, result.reconstruct())
print(json.dumps(payload, indent=2))
print("Reconstructed shape:", reconstructed.shape)
